# The judge

**A verdict you cannot reproduce is an opinion, and an opinion cannot gate a release.** Same tree in, same verdict digest out, every time.

> **Every cell in this notebook runs.** They are generated from
> [`tools/notebooks/spec.py`](../tools/notebooks/spec.py) and executed by CI, so a
> cell that cannot run does not reach a commit. Change a cell, re-run it, and the
> page is yours — that is what it is for.


This is not a linter with opinions and not a model asked to review. It projects
an AST into the same graph everything else uses, then runs deterministic queries
over it. Which means an audit finding *composes*: `audit | impact` answers "what
else does this violation reach", which no standalone linter can do.

The load-bearing verdict is **`INDETERMINATE`**. A file that fails to parse, a
dynamic import, a `getattr` call target — the judge says it could not decide
rather than guessing either way. A judge that silently reports `UPHELD` for what
it could not examine is worse than no judge, because the green result is now a
lie.

In [ ]:
# --- setup: works locally, on Binder, and on Colab -------------------------
import subprocess, sys, pathlib

def _ensure_installed():
    """Install the package if it is not importable. No-op when it already is."""
    try:
        import slpie, gratimos          # noqa: F401
        return pathlib.Path(slpie.__file__).parent.parent
    except ModuleNotFoundError:
        pass
    here = pathlib.Path.cwd()
    root = next(
        (p for p in [here, *here.parents] if (p / "pyproject.toml").exists()), None,
    )
    if root is None:                     # Colab: no checkout, so fetch one
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/Reimain/Macropol-s.git", "/content/Macropol-s"],
            check=True,
        )
        root = pathlib.Path("/content/Macropol-s")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(root)],
                   check=True)
    sys.path.insert(0, str(root))
    return root

ROOT = _ensure_installed()
print("package root:", ROOT)

import slpie
print("slpie", slpie.__version__)

## Judge this repository

In [ ]:
from slpie.audit import audit_self

verdict = audit_self()

print("rules run:  ", len(verdict.judgements))
print("coverage:   ", f"{verdict.coverage:.0%}")
print("undecided:  ", verdict.indeterminate)
print("clean:      ", verdict.clean)
print("digest:     ", verdict.digest)

## The verdicts

In [ ]:
from slpie.audit import Verdict

for judgement in verdict.judgements:
    mark = {Verdict.UPHELD: "✓", Verdict.VIOLATED: "✕",
            Verdict.INDETERMINATE: "?", Verdict.INAPPLICABLE: "–"}[judgement.verdict]
    print(f"  {mark} {judgement.verdict.value:14} {judgement.rule:22} {judgement.subject[:30]}")

## Reproducible by construction

Two runs over an unchanged tree produce an identical digest. That is the single value CI pins — "the architecture has not drifted" becomes a string comparison.

In [ ]:
again = audit_self()
print("first: ", verdict.digest)
print("second:", again.digest)
print("identical:", verdict.digest == again.digest)

## Honest about what it could not see

`INDETERMINATE` counts **against** coverage, reported alongside the verdicts rather than hidden.

In [ ]:
undecided = [j for j in verdict.judgements if j.verdict is Verdict.INDETERMINATE]
print(f"{len(undecided)} rule(s) could not be decided\n")
for judgement in undecided:
    print(f"  {judgement.subject}")
    print(f"    {judgement.detail[:70]}")
    for item in judgement.evidence[:1]:
        print(f"    at {item.location.uri.rsplit('/', 1)[-1]}:{item.location.line}")
    print()

## Catch a deliberate violation

In [ ]:
import pathlib, tempfile
from slpie.audit import Check, audit

WORK = pathlib.Path(tempfile.mkdtemp(prefix="slpie-nb-"))
fake = WORK / "slpie"
fake.mkdir()
(fake / "__init__.py").write_text("")
(fake / "rogue.py").write_text("import gratimos\n")       # <- breaks invariant 8

result = audit(WORK, checks=[Check("single-import", {
    "rule": "slpie→gratimos", "ring": "slpie", "target": "gratimos",
    "allowed": "slpie.artifacts.codegen",
})])

for judgement in result.judgements:
    print(f"  {judgement.verdict.value:12} {judgement.subject}")
    print(f"    {judgement.detail[:74]}")
    for item in judgement.evidence[:2]:
        print(f"    evidence: {item.location.uri.rsplit('/', 1)[-1]}:{item.location.line}")

Every verdict carries **file and line**, always. A verdict without evidence would be an assertion, and this whole module exists to not make assertions.

## As a CI gate

In [ ]:
from slpie.compose import Composition, Context, registry

verbs = registry()
violations = Composition.read("audit | verdicts --only violated", verbs=verbs).run(
    Context(root=str(ROOT)),
)
print("violations in this repository:", violations.flow.size)
print()
print("exit code discipline: 0 clean, 3 findings at or above --fail-on,")
print("so `slpie 'audit | verdicts' --fail-on high` is a usable gate.")

## It is checked by the mechanism it replaces

`tests/test_slpie_boundaries.py` is not deleted or weakened. It walks the tree with `ast` independently, and the suite asserts that the two **agree**. If the judge ever reported `UPHELD` for something the test catches, the judge is what is broken — and the suite says so.

In [ ]:
# Scratch cell — audit any tree you like.
target = ROOT / "gratimos"
result = audit(target)
print(f"{target.name}: {len(result.judgements)} judgement(s), "
      f"coverage {result.coverage:.0%}, digest {result.digest[:16]}")